# Simple base-rate merged results

Explore `data/simple/simple_merged_results.csv` from a benchmark run (`benchmark/simple-benchmark.ipynb`).

Each row has **`score`** (`true`/`false`): whether the parsed answer matches normative **P(C|T)**. **`path_c_confusion`** flags answers matching **P(T|C)** (the inverse-conditional lure).

In [34]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MERGED_DIR = ROOT / "data" / "simple"
MERGED_CSV = None
for name in (
    "simple_merged_results (2).csv",
):
    candidate = MERGED_DIR / name
    if candidate.is_file():
        MERGED_CSV = candidate
        break
if MERGED_CSV is None:
    raise FileNotFoundError(
        f"Missing merged results under {MERGED_DIR}. Run the simple benchmark first "
        "(benchmark/simple-benchmark.ipynb)."
    )

df = pd.read_csv(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
else:
    raise KeyError("Merged CSV must include 'score'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")
else:
    df["parseable_bool"] = True

if "path_c_confusion" in df.columns:
    df["path_c_confusion_bool"] = (
        df["path_c_confusion"].astype(str).str.lower().eq("true")
    )
else:
    df["path_c_confusion_bool"] = False

df["score_true"] = df["score_value"].astype(bool)

VARIANT_ORDER = ["open_probs", "mc_numeric_probs"]

print("Loaded:", MERGED_CSV)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
print(
    "Normative pass:", int(df["score_true"].sum()), "/", len(df),
    "| P(T|C) confusion:", int(df["path_c_confusion_bool"].sum()), "/", len(df),
)
df.head()

Loaded: c:\src2\sceptical-llms\data\simple\simple_merged_results (2).csv
Rows: 20
Models: ['google/gemini-2.5-flash']
Vignettes: 10
Normative pass: 5 / 20 | P(T|C) confusion: 6 / 20


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_choice,parsed_confidence,scoring_type,parseable,score,path_c_confusion,score_value,parseable_bool,path_c_confusion_bool,score_true
0,actor_waiter_overlap__mc_numeric_probs,actor waiter overlap,well_posed,0,mc,True,mc_numeric_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force, 0.00% are also hol...",True,well_posed,...,D,NaN,mc_numeric,True,False,False,0,True,False,False
1,actor_waiter_overlap__open_probs,actor waiter overlap,well_posed,0,open,True,open_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US labor force, 0.00% are also hol...",True,well_posed,...,NaN,NaN,open,True,False,False,0,True,False,False
2,ca_trump_voter__mc_numeric_probs,CA Trump voter,well_posed,0,mc,True,mc_numeric_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 7.8% are vot...",True,well_posed,...,NaN,NaN,mc_numeric,False,False,False,0,False,False,False
3,ca_trump_voter__open_probs,CA Trump voter,well_posed,0,open,True,open_probs,"You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered voters, 7.8% are vot...",True,well_posed,...,NaN,NaN,open,True,False,False,0,True,False,False
4,college_stem_work__mc_numeric_probs,college STEM work,well_posed,0,mc,True,mc_numeric_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong undergraduate students enrolled at...,True,well_posed,...,E,NaN,mc_numeric,True,False,True,0,True,True,False


In [35]:
def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, normative score, and P(T|C) confusion by group."""
    work = df.copy()
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "path_c_confusion": grouped["path_c_confusion_bool"].sum(),
            "score_rate": grouped["score_value"].mean(),
            "path_c_rate": grouped["path_c_confusion_bool"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)
    summary["path_c_pct"] = (summary["path_c_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])
    return summary


score_summary_table("variant", order=VARIANT_ORDER)

,n,score_true,score_false,unparseable,path_c_confusion,score_rate,path_c_rate,score_pct,path_c_pct
variant,,,,,,,,,
open_probs,10,0,10,0,5,0.0,0.5,0.0,50.0
mc_numeric_probs,10,5,4,1,1,0.5,0.1,50.0,10.0


In [36]:
score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)

,n,score_true,score_false,unparseable,path_c_confusion,score_rate,path_c_rate,score_pct,path_c_pct
vignette_name,,,,,,,,,
CA Trump voter,2,0,1,1,0,0.0,0.0,0.0,0.0
actor waiter overlap,2,0,2,0,0,0.0,0.0,0.0,0.0
college STEM work,2,0,2,0,2,0.0,1.0,0.0,100.0
covid vaccine (blue/red),2,1,1,0,1,0.5,0.5,50.0,50.0
diabetes insulin obese,2,1,1,0,1,0.5,0.5,50.0,50.0
discharged weapon (last year),2,1,1,0,0,0.5,0.0,50.0,0.0
english teacher humanities,2,0,2,0,1,0.0,0.5,0.0,50.0
healthcare employment,2,0,2,0,1,0.0,0.5,0.0,50.0
military overseas (federal pool),2,1,1,0,0,0.5,0.0,50.0,0.0


## `mc_numeric_probs` detail

MC options, parsed letter, normative letter, score, and whether the choice is the **P(T|C)** lure.

In [37]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]
MC_LURE_COLS = [f"option_{letter}_lure" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, label_col, lure_col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS, MC_LURE_COLS):
        label = row.get(label_col)
        lure = row.get(lure_col)
        if pd.notna(label) and str(label).strip():
            lure_text = f" [{lure}]" if pd.notna(lure) and str(lure).strip() else ""
            parts.append(f"{letter}: {label}{lure_text}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "normative_choice",
        "p_t_given_c",
        "score",
        "score_value",
        "path_c_confusion",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 160)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,normative_choice,p_t_given_c,score,score_value,path_c_confusion,answer_line
2,CA Trump voter,A: About 42% [Bayes P(C|T)] | B: About 8% [P(T|C)*P(T|D) confusion] | C: About 31% [P(T|C) confusion] | D: About 27% [P(T|D) confusion] | E: About 5% [P(C) ...,NaN,A,0.310,False,0,False,$P(T
0,actor waiter overlap,A: About 12% [Bayes P(C|T)] | B: About 55% [P(T|D) confusion] | C: About 36% [P(T|C)*P(T|D) confusion] | D: About 0% [P(D) confusion] | E: About 65% [P(T|C)...,D,A,0.650,False,0,False,D
4,college STEM work,A: About 24% [Bayes P(C|T)] | B: About 63% [P(T|C)*P(T|D) confusion] | C: About 74% [P(T|D) confusion] | D: About 17% [P(D) confusion] | E: About 85% [P(T|C...,E,A,0.850,False,0,True,P(R|E) = 0.74
6,covid vaccine (blue/red),A: About 66% [Bayes P(C|T)] | B: About 1% [P(T|C)*P(T|D) confusion] | C: About 8% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 19% [P(C) ...,A,A,0.080,True,1,False,A
8,diabetes insulin obese,A: About 43% [Bayes P(C|T)] | B: About 5% [P(D) confusion] | C: About 16% [P(T|D) confusion] | D: About 20% [P(T|C) confusion] | E: About 3% [P(T|C)*P(T|D) ...,A,A,0.200,True,1,False,A
10,discharged weapon (last year),A: About 77% [Bayes P(C|T)] | B: About 0% [P(T|D) confusion] | C: About 13% [P(D) confusion] | D: About 30% [P(C) confusion],A,A,0.003,True,1,False,A
12,english teacher humanities,A: About 52% [Bayes P(C|T)] | B: About 69% [P(T|C) confusion] | C: About 55% [P(T|D) confusion] | D: About 0% [P(D) confusion] | E: About 38% [P(T|C)*P(T|D)...,D,A,0.690,False,0,False,D
14,healthcare employment,A: About 9% [Bayes P(C|T)] | B: About 10% [P(D) confusion] | C: About 32% [P(T|C)*P(T|D) confusion] | D: About 60% [P(T|D) confusion] | E: About 54% [P(T|C)...,H,A,0.540,False,0,False,- $P(H|P) = 0.54$
16,military overseas (federal pool),A: About 38% [Bayes P(C|T)] | B: About 37% [P(T|C)*P(T|D) confusion] | C: About 58% [P(T|C) confusion] | D: About 64% [P(T|D) confusion] | E: About 20% [P(D...,A,A,0.580,True,1,False,* P(A) = 0.14 (14% are US Army active-duty service members)
18,professional drivers speeding,A: About 85% [Bayes P(C|T)] | B: About 0% [P(D) confusion] | C: About 16% [P(T|C) confusion] | D: About 10% [P(T|D) confusion] | E: About 2% [P(T|C)*P(T|D) ...,A,A,0.160,True,1,False,A


## `open_probs` detail

Re-parse open responses and compare to normative **P(C|T)** and **P(T|C)**.

In [38]:
from benchmarks.base_rate import matches_scepticism_target, parse_open_response
from benchmarks.simple_rate import PATH_C_LURE_NAME, load_benchmark, matches_path_c_confusion

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    return pd.Series(
        {
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": matches_scepticism_target(item, parsed),
            "path_c_confusion_rescored": matches_path_c_confusion(item, parsed),
            "p_t_given_c_pct": float(row["p_t_given_c"]) * 100,
        }
    )


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

idx = open_probs.index
df.loc[idx, "parsed_percent"] = pd.to_numeric(
    open_probs["parsed_percent_rescored"], errors="coerce"
)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
df.loc[idx, "path_c_confusion_bool"] = open_probs["path_c_confusion_rescored"].astype(bool)

open_probs_view = open_probs[
    [
        "example_id",
        "vignette_name",
        "normative_percent",
        "normative_open",
        "p_t_given_c_pct",
        "parsed_numbers",
        "parsed_percent_rescored",
        "score_true",
        "path_c_confusion_rescored",
    ]
].sort_values("vignette_name")

print(
    "Rescored open_probs normative pass:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
    "| P(T|C) confusion:",
    int(open_probs["path_c_confusion_rescored"].sum()),
    "/",
    len(open_probs),
)
open_probs_view

Rescored open_probs normative pass: 0 / 10 | P(T|C) confusion: 5 / 10


,example_id,vignette_name,normative_percent,normative_open,p_t_given_c_pct,parsed_numbers,parsed_percent_rescored,score_true,path_c_confusion_rescored
3,ca_trump_voter__open_probs,CA Trump voter,42.100,42%,31.0,"[7.8, 4.9]",4.9,False,False
1,actor_waiter_overlap__open_probs,actor waiter overlap,12.190,12%,65.0,[0.0],0.0,False,False
5,college_stem_work__open_probs,college STEM work,23.850,24%,85.0,"[4.6, 17.0, 85.0]",85.0,False,True
7,covid_vaccine_blue_red__open_probs,covid vaccine (blue/red),66.200,66%,8.0,"[19.0, 7.8, 8.0, 10.0]",10.0,False,True
9,diabetes_insulin_obese__open_probs,diabetes insulin obese,42.810,43%,20.0,"[9.0, 3.2, 5.3, 20.0]",20.0,False,True
11,discharged_weapon_last_year__open_probs,discharged weapon (last year),77.270,77%,0.3,"[30.0, 13.0]",13.0,False,False
13,english_teacher_humanities__open_probs,english teacher humanities,52.060,52%,69.0,"[0.11, 0.13, 69.0, 11.0, 13.0, 60.0]",60.0,False,True
15,healthcare_employment__open_probs,healthcare employment,9.091,9.1%,54.0,"[1.0999999999999999, 9.9, 54.0, 60.0]",60.0,False,True
17,military_overseas_federal_pool__open_probs,military overseas (federal pool),38.350,38%,58.0,"[14.000000000000002, 20.0, 50.0]",50.0,False,False
19,professional_drivers_speeding__open_probs,professional drivers speeding,85.490,85%,16.0,"[1.3, 0.35, 0.35000000000000003, 35.0]",35.0,False,False


## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for all 10 vignettes. **Normative** = P(C|T) (`normative_percent`). **P(T|C)** = `p_t_given_c` (inverse-conditional lure).

In [39]:
from benchmarks.base_rate import parse_open_response, parse_response
from benchmarks.simple_rate import PATH_C_LURE_NAME
from scripts.build_base_rate_prompts import _load_overlap

OVERLAP_VIGNETTE_NAMES = {v.name for v in _load_overlap()}

items_meta = pd.read_csv(ROOT / "data" / "simple" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter.lower()}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}: {label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_lure = item_mc.get(f"option_{mc_choice.lower()}_lure", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    path_c_pct = float(item_open["p_t_given_c"]) * 100
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "overlap": vignette_name in OVERLAP_VIGNETTE_NAMES,
            "normative_open": item_open["normative_open"],
            "normative_pct": normative_pct,
            "p_t_given_c_pct": path_c_pct,
            "open_parsed_pct": open_pct,
            "open_delta_vs_norm_pp": None if open_pct is None else open_pct - normative_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "open_path_c": bool(open_row.get("path_c_confusion_bool", False)),
            "mc_choices": _format_mc_choices(item_mc),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_lure": mc_lure,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_norm_pp": None if mc_pct is None else mc_pct - normative_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "mc_path_c": mc_lure == PATH_C_LURE_NAME or (
                mc_pct is not None and abs(mc_pct - path_c_pct) <= 0.5
            ),
            "normative_mc_letter": item_mc["normative_choice"],
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
    "| open P(T|C) confusion:",
    int(open_vs_mc["open_path_c"].sum()),
    "| mc P(T|C) confusion:",
    int(open_vs_mc["mc_path_c"].sum()),
)

COMPARISON_COLUMNS = [
    "vignette_name",
    "overlap",
    "normative_pct",
    "open_parsed_pct",
    "open_score",
    "mc_choices",
    "mc_choice",
    "mc_label",
    "mc_score",
]

pd.set_option("display.max_colwidth", 160)
display(open_vs_mc[COMPARISON_COLUMNS])

open_probs pass: 0 / 10 | mc_numeric_probs pass: 5 / 10 | open P(T|C) confusion: 5 | mc P(T|C) confusion: 1


,vignette_name,overlap,normative_pct,open_parsed_pct,open_score,mc_choices,mc_choice,mc_label,mc_score
0,CA Trump voter,False,42.100,4.9,False,A: About 42% | B: About 8% | C: About 31% | D: About 27% | E: About 5%,,,False
1,actor waiter overlap,True,12.190,0.0,False,A: About 12% | B: About 55% | C: About 36% | D: About 0% | E: About 65%,D,About 0%,False
2,college STEM work,True,23.850,85.0,False,A: About 24% | B: About 63% | C: About 74% | D: About 17% | E: About 85%,E,About 85%,False
3,covid vaccine (blue/red),False,66.200,10.0,False,A: About 66% | B: About 1% | C: About 8% | D: About 10% | E: About 19%,A,About 66%,True
4,diabetes insulin obese,True,42.810,20.0,False,A: About 43% | B: About 5% | C: About 16% | D: About 20% | E: About 3%,A,About 43%,True
5,discharged weapon (last year),False,77.270,13.0,False,A: About 77% | B: About 0% | C: About 13% | D: About 30%,A,About 77%,True
6,english teacher humanities,True,52.060,60.0,False,A: About 52% | B: About 69% | C: About 55% | D: About 0% | E: About 38%,D,About 0%,False
7,healthcare employment,False,9.091,60.0,False,A: About 9% | B: About 10% | C: About 32% | D: About 60% | E: About 54%,H,NaN,False
8,military overseas (federal pool),False,38.350,50.0,False,A: About 38% | B: About 37% | C: About 58% | D: About 64% | E: About 20%,A,About 38%,True
9,professional drivers speeding,True,85.490,35.0,False,A: About 85% | B: About 0% | C: About 16% | D: About 10% | E: About 2%,A,About 85%,True


In [40]:
basically, whenever overlap is false, everything is correct; but when overlap is true, the open output is wrong, but the mc is correct.
COuld this be due to the MC being way to easy?

SyntaxError: invalid syntax (1242660209.py, line 1)

### Printable comparison

Per vignette: source probabilities from `items.csv` (P(C), P(D), P(T|C), P(T|D)), normative / open / MC answers, and full `mc_numeric_probs` prompt.

In [ ]:
import re

from benchmarks.base_rate import parse_open_response, parse_response

benchmark_df = pd.read_csv(ROOT / "data" / "simple" / "benchmark.csv")


def mc_numeric_options_prompt(prompt: str) -> str:
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(item_row: pd.Series) -> str:
    return " | ".join(
        [
            f"P(C)={float(item_row['p_c']):.6g}",
            f"P(D)={float(item_row['p_d']):.6g}",
            f"P(T|C)={float(item_row['p_t_given_c']):.6g}",
            f"P(T|D)={float(item_row['p_t_given_d']):.6g}",
        ]
    )


print_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(item_open),
            "normative_pct": float(item_open["normative_percent"]),
            "p_t_given_c_pct": float(item_open["p_t_given_c"]) * 100,
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'P(T|C)':>10} {'open':>10} {'MC label':>14}")
print("-" * 84)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%  (P(C|T))")
    print(f"  P(T|C):      {row.p_t_given_c_pct:.4g}%  (inverse-conditional lure)")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

## Optional: split by model when multiple LLMs are present

In [ ]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["path_c_confusion_bool"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")